# 2. Vector search

Here the docs are compared based on semantic meaning rather than keyword search as in search engines previously discussed.

Topics covered:-
embeddings and end with persistent vector indexes (sqlitesearch, PGVector) and ONNX-based embedders for lightweight deployments

## Vector search

Keyword searches looks for exact match in docs.

So vector search mathces ideas instead of matching words.



### Vector search process

Has two stages:-
1. Offline where documnets are converted into vectors and stored them in index
2. Online where the query is embedded into a vector using the same model and then search is performed

An embedding model produces these vectors. Most common distance metric is cosine similarity and higher the value, more similar the docs are



## Keyword vs vector search

Keyword search looks for eaxct word match but vector search loooks for meaning.

Keyword search uses inverted index (TF-IDF). Vector search uses vector index based on cosine similarity

Vector search is better but it adds operational complexity. So **ADVICE** is start with text search and then move on to vector search when you feel its needed.

**BEST** use hybrid search which is a combination of both.

## Embeddings

The process of turning text into numbers is called embedding and the vectors we get back are also called embeddings.

In [1]:
from sentence_transformers import SentenceTransformer
embedding_model=SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
# try an example
q='Can i still joine the course?'
embedded_q_vector=embedding_model.encode(q)

The above code downaload the embedding model and tokenizer.

In [3]:
embedded_q_vector,embedded_q_vector.shape

(array([-1.84794376e-03, -5.72410338e-02, -4.24024789e-03,  1.88935474e-02,
         2.82832738e-02, -1.03440005e-02, -4.85474057e-02, -1.01650015e-01,
        -8.04928541e-02, -1.20822480e-02, -2.14720964e-02,  1.63812917e-02,
         8.73676129e-03,  2.34868508e-02,  1.44208567e-02, -2.95178238e-02,
        -6.86395317e-02, -4.49078530e-03, -4.76951413e-02, -1.07721975e-02,
        -8.64377916e-02,  2.64867749e-02, -7.39833862e-02,  4.30801958e-02,
        -3.05371545e-02,  1.56237427e-02,  3.16433273e-02,  8.64882991e-02,
        -1.50857968e-02, -7.59720653e-02, -2.69011967e-02,  4.93408702e-02,
        -1.32593177e-02,  3.74351554e-02,  1.95681863e-02, -3.78549448e-04,
         5.69494292e-02, -8.14097524e-02,  2.81400923e-02,  1.44231068e-02,
        -3.67073566e-02, -4.46327060e-04, -3.96147706e-02,  5.33254305e-03,
         9.19253938e-03,  8.46043006e-02, -5.30288592e-02, -8.51033852e-02,
         5.05385455e-04,  5.01306430e-02,  2.86293980e-02, -3.43284607e-02,
        -5.0

In [4]:
# creating a doc with similar content
doc='yes you can join the course'
embedded_doc_vector=embedding_model.encode(doc)

In [5]:
# check similarity
embedded_q_vector.dot(embedded_doc_vector)

np.float32(0.90842164)

In [6]:
# trying an unrelated query
doc='Hello. How are you?'
doc_embed=embedding_model.encode(doc)

In [7]:
embedded_q_vector.dot(doc_embed)

np.float32(0.020480271)

### Word embeddingd and sentence embedings

The model learns to place words in a multi-dimensional space. Words with similar meaning land close to each other.

When user asks a question, the question is embedded into the same space and then similarity is calculated.

The model encodes the whole sentence not words in isolation.
*So an embedding model takes text in and returns a fixed length array of numbers and it is trained in a way so that texts with similar meaning get similar vectors


The whole idea of vector search is similar docs get similar vectors and dot product between them tells their similarity.

### Cosine similarity

The model outputs normalized vectors and when we take dot product of tose vectors, it is nothing but cosine similarity



## Embedding the FAQ dataset

We turn the docs into embeddings of size (no of docs,no of dims)

So first get the load_faq function from te ingestion.py

In [8]:
from ingest import load_faq_data


In [9]:
docs=load_faq_data()

### Generate embeddings

Here we generate embeddings of docs by passing the docs ( but beofre that, we need to build one tetxt per doc and then ) in batches of 50 insated of padsssing all of them at once and append it into a list or so

In [10]:
# build one text per doct
texts=[]
for doc in docs:
    text=doc['question']+' '+doc['answer']
    texts.append(text)

In [11]:
texts[1]

"Course: What are the prerequisites for this course? To get the most out of this course, you should have:\n\n- Basic coding experience\n- Familiarity with SQL\n- Experience with Python (helpful but not required)\n\nNo prior data engineering experience is necessary. See [Readme on GitHub](https://github.com/DataTalksClub/data-engineering-zoomcamp/blob/main/README.md#prerequisites).\n\nIf you have these basics, you're ready to start — you don't need to master everything up front. The course covers Git and GitHub (see *How do I use Git/GitHub for this course?*), and you'll pick up the command-line/Linux basics you need during the setup modules."

In [12]:
# now generate embeddings
from tqdm.auto import tqdm
batch_size=50
embedded_docs=[]

for i in tqdm(range(0,len(texts),batch_size)):
    texts_batch=texts[i:i+50]
    batch_vectors=embedding_model.encode(texts_batch)
    embedded_docs.extend(batch_vectors)


  0%|          | 0/27 [00:00<?, ?it/s]

In [13]:
import numpy as np
X=np.array(embedded_docs)

In [14]:
X.shape
# (no of docs, no of embedding dims)

(1350, 384)

## Vector Search

## Scoring the docs

Process of vector search using numpy:-

We take the query, embed it and then take the dot product of the query against all docs. Then select the top 5 results using argsort and so.

**Therefore doing vector search by hand with numpy is fine but for large datsets, a library is better which handles filtering and reranking**

In [15]:
# when we get a query
query='When does the course start?'
embedded_query=embedding_model.encode(query)

In [16]:
# comapre it with alll docs and get similarity sUnicodeTranslateError
X.dot(embedded_query)

array([ 0.60896885,  0.31690836,  0.52076197, ..., -0.01416232,
        0.10315572,  0.08913746], shape=(1350,), dtype=float32)

In [17]:
# the above is a matrix vector multipication so we get a vector of size of docs
# therefore we need to sort them and pick the best 5 for our task
scores=X.dot(embedded_query)

In [18]:
scores.shape

(1350,)

In [19]:
scores_sorted=np.sort(scores)
scores_sorted[:5]

array([-0.15817106, -0.15109712, -0.12621185, -0.12618263, -0.11957842],
      dtype=float32)

In [20]:
# we use argsort because we also need the index postion of the top 5 scores to later retrieve the docs of those indicesabs
top5=np.argsort(scores)[-5:]# argsort sorts from lowest to hisghest so last 5 are the top ones

In [21]:
# since argsort sorts lowest to highest
top5=top5[::-1]

In [22]:
top5

array([546, 898, 443,   0, 912])

In [23]:
# in one line
top5=np.argsort(-scores)[:5]

In [24]:
scores[top5]

array([0.75943255, 0.6648307 , 0.6272896 , 0.60896885, 0.563987  ],
      dtype=float32)

In [25]:
# reading the actual top docs
for idx in top5:
    print(scores[idx])
    print(texts[idx])
    print()

0.75943255
When will the course be offered next? Summer 2027.

0.6648307
How long is the course? Approximately 4 months, but it may take longer if you want to engage in extra activities such as an additional project or writing an article.

0.6272896
What is the overall timeline and structure of the course? About 2.5-3 months: five lectures (each around 1.5 hours) with two-to-three weeks between them, followed by a roughly 3-week window for the capstone project.

0.60896885
Course: When does the course start? A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).

- Register via the link in the course repo before the cohort starts.
- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.
- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel.

0.563

Doing this by hand with numpy is fine for a small dataset. A larger one needs a library that also handles filtering and ranking. That's what we turn to next.

## Vector search with minsearch

VectorSearch API is similar to TextSearch API. This also has iption of filtering which plain numpy search didn't have

In [26]:
# creating the index
from minsearch import VectorSearch

vindex=VectorSearch(keyword_fields=['course'])

# fit the embeddings and also pass docs as payloadabs
vindex.fit(X,docs)


### Searching

In [27]:
# search for a query
query = "I just discovered the course. Can I still join it?"
query_vector=embedding_model.encode(query)

In [28]:
# search by passing the embedded_query, num-results
search_results=vindex.search(query_vector,num_results=5)

In [29]:
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '41aabbd7c5',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'},
 {'id': '2d8b16c2a0',
  'course': 'mlops-zoomcamp',
  'section':

In [30]:
search_results[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

### Filtering by course

In [31]:
search_results=vindex.search(
    query_vector,
    filter_dict={'course':'llm-zoomcamp'},
    num_results=5
)

In [32]:
search_results[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

## RAG with Vector Search

Here we only need to chnage or modify the search part to use the VectorSearch index and also here we can't send the query directly to search but send it after embedding the query.

This is done by subclassing the RAGBase class and only change the search part

In [33]:
from ollama import chat
from ollama import ChatResponse

In [34]:
from rag_helper import RAGBase

class RAGVector(RAGBase):
    def __init__(self,embedder,**kwargs):
        # here RAGVector class takes the 'embedder' variable and forwards the other keywrod arguments to teh parent RAGBase class
        super().__init__(**kwargs)
        self.embedder=embedder
    def search(self,query,num_results=5):
        query_vector=self.embedder.encode(query)
        filter_dict={'course':'llm-zoomcamp'}
        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

SyntaxError: invalid syntax (rag_helper.py, line 19)

The __init__ method adds one extra argument, embedder, for the sentence transformer. Inside search we use it to turn the query into a vector. Then we query vindex with that vector instead of the raw text. Everything else is inherited from RAGBase.

In [ ]:
from ollama import Client
client=Client()

In [ ]:
# using it
vector_assistant=RAGVector(
    embedder=embedding_model,
    index=vindex,
    llm_client=client
)

In [ ]:
# try it
vector_assistant.rag("the program has already begun, can I still sign up?")

In [ ]:
# try it
vector_assistant.rag("the program has already begun, can I still sign up?")

## Vector Search with SQLitesearch

The problem with minsearch was that we had to re index every time we start whereas in sqlitesearch, there is persistent stoarge, which means the embedded vectors stay on disk even after startup and we can use them later on and vectors are stored in SQLite db.

The advantages of sqlitesearch is:-
1. has persistent storage
2. uses *ANN* (Approximate Nearest Neighbours) which means, first the closest neighbours to query vector are found and then comparison is done between all closest vectors and query vector instead of comapring query vector with all doc vectors as in *NN*(Nearest Neighbours) and then get top ones
3. no need to rebuild index on startup

```
NN (exact):    compare query against ALL documents -> top 5
ANN (approx):  narrow down to a region -> compare within region -> top 5
```

### Creating index


In [ ]:
from sqlitesearch import VectorSearchIndex

vs_index=VectorSearchIndex(
    keyword_fields=['course'],
    mode='ivf',
    db_path='faq_vectors.db'
)

sqlitesearch supports three ANN modes:

- lsh (default): up to 100K vectors, random hyperplane projections
- ivf: 10K-500K vectors, K-means clustering
- hnsw: 10K-1M+ vectors, proximity graph (highest recall)
  
For our small dataset, lsh is fine. All modes use two-phase search: approximate candidate retrieval, then exact cosine similarity reranking.

### Indexing the data


In [ ]:
vs_index.fit(X,docs)

### Searching

First embed the query and then search

In [ ]:
query = "I just discovered the course. Can I still join it?"

In [ ]:
embedded_query=embedding_model.encode(query)

In [ ]:
search_results_sql=vs_index.search(
    embedded_query,
    num_results=5
)

In [ ]:
# Filtering by course
# Filtering works the same way:

results = vs_index.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [ ]:
# closing the connection
vs_index.close()

### Reopening the index

In [35]:
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

embedding_model=SentenceTransformer('all-MiniLM-L6-v2')
vs_index=VectorSearchIndex(
    keyword_fields=['course'],
    mode='ivf',
    db_path='faq_vectors.db'
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [36]:
query_vector = model.encode("How do I run Kafka?")
results = vs_index.search(query_vector, num_results=5)

NameError: name 'model' is not defined

### using SQLitesearch vector search in RAG

In [37]:
# load the vector store index 
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

embedding_model=SentenceTransformer('all-MiniLM-L6-v2')

vs_index=VectorSearchIndex(
    keyword_fields=['course'],
    mode='ivf',
    db_path='faq_vectors.db'
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
# we'll use the same class RAGBase but we'll subclass it and replace the search method of it and also include a embedder
from rag_helper import RAGBase
class RAGVector(RAGBase):
    def __init__(self,embedder,**kwargs):
        super().__init__(**kwargs)
        self.embedder=embedder

    def search(self,query,num_results=5):
        embedded_query=self.embedder(query)
        filter_dict={'course':'llm-zoomcamp'}
        return self.index.search(
            embedded_query,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [ ]:
vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=openai_client,
)

In [ ]:
vector_assistant.rag("the program has already begun, can I still sign up?")

https://github.com/DataTalksClub/llm-zoomcamp/blob/main/02-vector-search/lessons/07-sqlitesearch-vector.md#comparing-minsearch-and-sqlitesearch-for-vector-search

**Skipped this** https://github.com/DataTalksClub/llm-zoomcamp/blob/main/02-vector-search/lessons/08-pgvector.md

## ONXX Runtime instead of PyTorch

https://github.com/DataTalksClub/llm-zoomcamp/blob/main/02-vector-search/lessons/09-onnx-embedder.md#using-onnx-runtime-instead-of-pytorch

For development and experiments, sentence-transformers is fine. For production you want the lighter option.